# BRIGHTER Multilingual Emotion Classification - mBERT Baseline

Fine-tunes multilingual BERT on English, Swahili, and Ukrainian for multi-label emotion classification.
The official BRIGHTER `dev` split is held out until final evaluation (Step 4).
We report macro-F1 for trained languages and zero-shot transfer to German, Portuguese, and Yoruba.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT: Path = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.brighter_emotion_pipeline import EVAL_LANGUAGES
from src.brighter_emotion_pipeline import RunConfig
from src.brighter_emotion_pipeline import create_train_internal_dev_split
from src.brighter_emotion_pipeline import create_trainer
from src.brighter_emotion_pipeline import describe_training_data
from src.brighter_emotion_pipeline import ensure_directories
from src.brighter_emotion_pipeline import evaluate_languages
from src.brighter_emotion_pipeline import explain_lime_examples
from src.brighter_emotion_pipeline import load_model
from src.brighter_emotion_pipeline import load_tokenizer
from src.brighter_emotion_pipeline import plot_macro_f1_by_language
from src.brighter_emotion_pipeline import plot_per_label_f1_by_language
from src.brighter_emotion_pipeline import select_lime_examples
from src.brighter_emotion_pipeline import set_random_seed
from src.brighter_emotion_pipeline import tokenize_dataset
from src.brighter_emotion_pipeline import write_markdown_summary

## Steps 1 + 2: Prepare Data and Inspect Training Distribution

Only the official train split is loaded here.
We don't touch the official dev split until final evaluation.

In [2]:
config: RunConfig = RunConfig(output_dir=PROJECT_ROOT / "outputs", reports_dir=PROJECT_ROOT / "reports")
set_random_seed(seed=config.seed)
ensure_directories(config=config)

tokenizer = load_tokenizer(config=config)
splits = create_train_internal_dev_split(config=config)
overview_frame: pd.DataFrame = describe_training_data(config=config)
overview_frame

,language,language_name,rows,anger_positive_rate,disgust_positive_rate,fear_positive_rate,joy_positive_rate,sadness_positive_rate,surprise_positive_rate
0,eng,English,2764,0.120478,0.000000,0.582489,0.243849,0.316932,0.303546
1,swa,Swahili,3307,0.094043,0.071969,0.028122,0.133958,0.105836,0.162080
2,ukr,Ukrainian,2466,0.039740,0.034874,0.069749,0.167072,0.135036,0.079481


Sanity check: all three training languages (eng, swa, ukr) should appear and label frequencies should look plausible (not zero, not near 100%). Open `reports/tables/training_data_overview.csv` to verify.

## Step 3: Train on Internal Train, Validate on Internal Dev

In [3]:
model = load_model(config=config)
tokenized_train_dataset = tokenize_dataset(dataset=splits["train"], tokenizer=tokenizer, config=config)
tokenized_internal_dev_dataset = tokenize_dataset(dataset=splits["internal_dev"], tokenizer=tokenizer, config=config)
trainer = create_trainer(
    model=model,
    tokenizer=tokenizer,
    tokenized_train_dataset=tokenized_train_dataset,
    tokenized_internal_dev_dataset=tokenized_internal_dev_dataset,
    config=config,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [4]:
trainer.train()
trainer.save_model(str(config.model_dir))
tokenizer.save_pretrained(config.model_dir)

Epoch,Training Loss,Validation Loss,Macro F1,Micro F1,Samples F1,F1 Anger,F1 Disgust,F1 Fear,F1 Joy,F1 Sadness,F1 Surprise
1,1.135020,1.020990,0.375533,0.397002,0.314477,0.212644,0.136364,0.657188,0.408907,0.416397,0.421702
2,0.967002,0.991864,0.402434,0.440236,0.331122,0.237805,0.147239,0.682274,0.496429,0.474093,0.376766
3,0.756124,0.994864,0.429391,0.472003,0.349452,0.271930,0.165049,0.683250,0.489051,0.505525,0.461538


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

[transformers] There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.encoder.layer.0.output.LayerNorm.beta', 'bert.encoder.layer.0.output.LayerNorm.gamma', 'bert.encoder.layer.1.attention.output.LayerNorm.beta', 'bert.encoder.layer.1.attention.output.LayerNorm.gamma', 'bert.encoder.layer.1.output.LayerNorm.beta', 'bert.encoder.layer.1.output.LayerNorm.gamma', 'bert.encoder.layer.2.attention.output.LayerNorm.beta', 'bert.encoder.layer.2.attention.output.LayerNorm.gamma', 'bert.encoder.layer.2.output.LayerNorm.beta', 'bert.encoder.layer.2.output.LayerNorm.gamma', 'bert.encoder.layer.3.attention.output.LayerNorm.beta', 'bert.encoder.layer.3.attention.output.LayerNorm.gamma', 'bert.encoder.layer.3.output.LayerNorm.beta', 'bert.encoder.layer.3.output.LayerNorm.gamma', 'bert.encoder.layer.4.attention.

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/home/nik/code/github.com/MayNiklas/MMD2-project/outputs/model/tokenizer_config.json',
 '/home/nik/code/github.com/MayNiklas/MMD2-project/outputs/model/tokenizer.json')

Check that internal dev macro-F1 is logged at each epoch. After training, `outputs/model` should contain the saved model and tokenizer.

## Final Evaluation on Official Dev

This is the first time we load the official dev split.

In [5]:
metrics_frame, per_label_metrics_frame, predictions_frame = evaluate_languages(
    trainer=trainer,
    tokenizer=tokenizer,
    languages=EVAL_LANGUAGES,
    split="dev",
    config=config,
)
metrics_frame[["language_name", "training_status", "macro_f1", "micro_f1", "samples_f1"]]

,language_name,training_status,macro_f1,micro_f1,samples_f1
0,English,trained,0.523449,0.656331,0.572836
1,Swahili,trained,0.247537,0.272660,0.216152
2,Ukrainian,trained,0.412169,0.460705,0.297992
3,German,zero-shot,0.320112,0.353678,0.291254
4,Brazilian Portuguese,zero-shot,0.286970,0.391304,0.306833
5,Yoruba,zero-shot,0.185321,0.262270,0.206439


In [6]:
per_label_metrics_frame

,language,language_name,split,training_status,label,f1
0,eng,English,dev,trained,anger,0.486486
1,eng,English,dev,trained,disgust,0.000000
2,eng,English,dev,trained,fear,0.691729
3,eng,English,dev,trained,joy,0.571429
4,eng,English,dev,trained,sadness,0.716049
5,eng,English,dev,trained,surprise,0.675000
6,swa,Swahili,dev,trained,anger,0.215385
7,swa,Swahili,dev,trained,disgust,0.206522
8,swa,Swahili,dev,trained,fear,0.113208
9,swa,Swahili,dev,trained,joy,0.370370


Compare the trained-language rows (eng/swa/ukr) against the zero-shot rows (deu/ptbr/yor). Full results are saved to `reports/tables/dev_language_metrics.csv` and `reports/tables/dev_per_label_metrics.csv`.

## Step 4: Graphs

In [7]:
macro_f1_path: Path = plot_macro_f1_by_language(metrics_frame=metrics_frame, config=config)
per_label_f1_path: Path = plot_per_label_f1_by_language(
    per_label_metrics_frame=per_label_metrics_frame,
    config=config,
)
macro_f1_path, per_label_f1_path

(PosixPath('/home/nik/code/github.com/MayNiklas/MMD2-project/reports/figures/macro_f1_by_language.png'),
 PosixPath('/home/nik/code/github.com/MayNiklas/MMD2-project/reports/figures/per_label_f1_by_language.png'))

Both plots are saved to `reports/figures/`. Check that axes are labeled and there is a legend.

## Step 5: LIME Error Analysis

In [ ]:
lime_examples_frame: pd.DataFrame = select_lime_examples(predictions_frame=predictions_frame, max_examples=6)
lime_paths: list[Path] = explain_lime_examples(
    model=trainer.model,
    tokenizer=tokenizer,
    examples_frame=lime_examples_frame,
    config=config,
)

for html_path in lime_paths:
    png_path = html_path.with_suffix(".png")
    if png_path.exists():
        from IPython.display import Image, display
        display(Image(filename=str(png_path)))

lime_paths

The HTML files in `reports/lime/` show which tokens the model attends to for each prediction. Open one to check that the highlighted words look reasonable.

## Step 6: Summary

In [9]:
summary_path: Path = write_markdown_summary(
    metrics_frame=metrics_frame,
    per_label_metrics_frame=per_label_metrics_frame,
    lime_paths=lime_paths,
    config=config,
)
summary_path

PosixPath('/home/nik/code/github.com/MayNiklas/MMD2-project/reports/results_summary.md')